# Ferienakademie 2026 — shared project playground

The global rules and scenario are fixed. The main things to change are `setup()` and `act()`.

The example below is intentionally simple and inefficient. Agents start randomly across a fairly large open playground. They explore until they find the cargo, and while the target is not visible from the cargo they push in their current exploration direction. Only once the target becomes locally visible does transport become directed.


In [ ]:
from fa2026 import Action, Cell, Config, Pheromone, execute, load_scenario

scenario = load_scenario("playground")
scenario.rules


## 1. Define the strategy

`setup(rules)` is called once. `act(...)` is called once per agent and turn. Agents know only local information; they have no absolute position, turn number, or agent ID.


In [ ]:
import math
import numpy as np


def setup(rules):
    return Config(
        pheromones=(Pheromone(decay=0.08, color="blue"),),
        initial_memory=((0.0, 0.0, 0.0, 0.0),),
        parameters=((0.55, 0.90, 0.10),),  # move step, push strength, emission
    )


def visible_vector(observation, flag):
    """Approximate vector to the center of all visible cells carrying flag."""
    mask = (observation.vision & int(flag)) != 0
    ys, xs = np.nonzero(mask)
    if xs.size == 0:
        return None
    center = observation.vision.shape[0] // 2
    dx = xs - center + 0.5 - observation.cell_position[0]
    dy = ys - center + 0.5 - observation.cell_position[1]
    return np.array([float(np.mean(dx)), float(np.mean(dy))])


def unit(vector):
    length = float(np.linalg.norm(vector))
    return vector / length if length > 1e-12 else np.zeros(2)


def act(observation, memory, agent_type, config):
    memory = memory.copy()
    step, push_strength, emission = config.parameters[agent_type]
    cargo = visible_vector(observation, Cell.CARGO)
    target = visible_vector(observation, Cell.TARGET)

    # Give every agent a persistent exploration direction. Initial fractional
    # positions break symmetry without exposing an agent ID or absolute position.
    if memory[2] < 0.5:
        phase = (
            observation.cell_position[0]
            + 0.61803398875 * observation.cell_position[1]
        ) % 1.0
        memory[0] = 2 * math.pi * phase
        memory[1] = 11.0
        memory[2] = 1.0

    angle = float(memory[0])
    countdown = float(memory[1]) - 1.0
    if countdown <= 0:
        angle = (angle + 2.399963229728653) % (2 * math.pi)
        countdown = 11.0
    explore = np.array([math.cos(angle), math.sin(angle)])

    # Once approximately inside the cargo, push. Before the target is visible,
    # the push direction is just the current exploration direction.
    on_cargo = cargo is not None and abs(cargo[0]) <= 1.05 and abs(cargo[1]) <= 1.05
    if on_cargo:
        push_direction = unit(target - cargo) if target is not None else explore
        memory[0] = math.atan2(push_direction[1], push_direction[0])
        memory[1] = countdown
        return Action(
            move=tuple(step * push_direction),
            push=tuple(push_strength * push_direction),
            pheromones=(emission,),
        ), memory

    # Approach visible cargo and leave a simple recruitment trail.
    if cargo is not None:
        return Action(
            move=tuple(step * unit(cargo)),
            pheromones=(emission,),
        ), memory

    # Follow a local pheromone maximum if one is nearby.
    pheromone = observation.pheromones[0]
    if float(np.max(pheromone)) > 1e-8:
        py, px = np.unravel_index(np.argmax(pheromone), pheromone.shape)
        direction = np.array([px - 1, py - 1], dtype=float)
        if np.linalg.norm(direction) > 0:
            return Action(move=tuple(step * unit(direction))), memory

    # Otherwise keep exploring, with only crude wall avoidance.
    move = step * explore
    center = observation.vision.shape[0] // 2
    sx = 1 if move[0] > 0 else (-1 if move[0] < 0 else 0)
    sy = 1 if move[1] > 0 else (-1 if move[1] < 0 else 0)
    if sx and int(observation.vision[center, center + sx]) & int(Cell.WALL):
        move[0] *= -1
    if sy and int(observation.vision[center + sy, center]) & int(Cell.WALL):
        move[1] *= -1

    memory[0] = math.atan2(move[1], move[0])
    memory[1] = countdown
    return Action(move=tuple(move)), memory


## 2. Watch one run

Display settings do not change the game. `draw_every` only controls how often the evolving state is redrawn.


In [ ]:
result = execute(
    scenario,
    setup,
    act,
    visualize=True,
    delay=0.0,
    draw_every=10,
    max_turns=1000,
)

if result.success:
    print(f"Success after {result.turns} turns.")
else:
    print(f"Not solved after {result.turns} turns.")


## 3. Run without visualization

Use this mode for fast testing. The score is the number of turns needed to place the cargo completely inside the target. Timing a run is also useful when trying many controller variants.


In [ ]:
from time import perf_counter

start = perf_counter()
result = execute(
    scenario,
    setup,
    act,
    visualize=False,
    max_turns=1000,
)
elapsed = perf_counter() - start

print(result)
print(f"{elapsed:.2f} s  |  {result.turns / elapsed:.0f} turns/s")


## Your turn

The baseline is deliberately weak. It does not know the target direction until target cells become locally visible from the cargo. More difficult scenarios can change the fixed rules and geometry while keeping the same `setup()` / `act()` interface. The goal is to discover local rules that solve tasks quickly and robustly.
